In [105]:
import pandas as pd
import numpy as np

In [106]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
df.head(2)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1


In [107]:
num_cols = list(df.select_dtypes('number'))
for c in num_cols:
    df[c] = df[c].fillna(0)

df = df.fillna('NA')

In [108]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [109]:
df.corr(numeric_only=True)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [110]:
from sklearn.model_selection import train_test_split

In [111]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [112]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [113]:
del df_train['converted']
del df_test['converted']
del df_val['converted']

In [114]:
from sklearn.metrics import mutual_info_score

def mutual_converted_info_score(series):
    return round(mutual_info_score(series, y_train),2)

In [115]:
str_cats = list(df.select_dtypes('string'))
mi = df_train[str_cats].apply(mutual_converted_info_score)
mi

lead_source          0.03
industry             0.00
employment_status    0.02
location             0.00
dtype: float64

In [116]:
from sklearn.feature_extraction import DictVectorizer
def get_Xs(df_train, df_val):
    dv = DictVectorizer(sparse=False)

    dicts_train = df_train.to_dict(orient='records')
    dicts_val = df_val.to_dict(orient='records')

    X_train = dv.fit_transform(dicts_train)
    X_val = dv.transform(dicts_val)
    return X_train, X_val


In [124]:
from sklearn.linear_model import LogisticRegression
def get_model(X_train, y_train, C=1):
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return model

def get_accuracy(model, X_val, y_val):
    y_pred = model.predict_proba(X_val)[:,1]
    decision = (y_pred>=0.5)
    accuracy = (y_val==decision).mean()
    return accuracy


In [125]:
X_train, X_val = get_Xs(df_train, df_val)
model = get_model(X_train, y_train)
original_accuracy = get_accuracy(model, X_val, y_val)
rounded_original_accuracy = round(original_accuracy,2)
print(rounded_original_accuracy)

0.64


In [130]:
numerical = list(df_train.select_dtypes('number'))
categorical = list(df_train.select_dtypes('string'))
features = numerical + categorical
for f in features:
    fs = features.copy()
    fs = [c for c in fs if c!=f]
    X_train, X_val = get_Xs(df_train[fs], df_val[fs])
    model = get_model(X_train, y_train)
    accuracy = get_accuracy(model, X_val, y_val)
    difference = round(original_accuracy - accuracy,3)
    print(f"{f}: {difference}")

annual_income: -0.079
number_of_courses_viewed: 0.002
interaction_count: 0.044
lead_score: 0.001
lead_source: 0.003
industry: 0.0
employment_status: 0.0
location: 0.0


In [131]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
for c in c_values:
    X_train, X_val = get_Xs(df_train, df_val)
    model = get_model(X_train, y_train, c)
    accuracy = get_accuracy(model, X_val, y_val)
    rounded_accuracy = round(accuracy,3)
    print(f"{c}: {rounded_accuracy}")

1e-06: 0.598
1e-05: 0.598
0.0001: 0.613
0.001: 0.645
